# Chapter 2
I will be using The Metamorphosis by Franz Kafka.

**-Perform version checks**

In [67]:
from importlib.metadata import version

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

torch version: 2.5.1
tiktoken version: 0.14.0


**-Load the story's text file**

In [68]:
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()
    
print("Total # of characters:", len(raw_text))
print(raw_text[:99])

Total # of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


**-Now split based on whitespace and punctuation/symbols, deleting all empty tokens**

In [69]:
import re
preprocessed = re.split(r'([,.:;?_!"()\'-]|--|\s)', raw_text)
preprocessed = [item.strip() for item in preprocessed if item.strip()]
print(len(preprocessed)) #24403
print(preprocessed[:30])

4863
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '-', '-', 'though', 'a', 'good', 'fellow', 'enough', '-', '-', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that']


**Note that in order to convert tokens into token IDs, we build a vocabulary to map each unique word to a unique integer value.**

**-We create a vocabulary where each unique token is added in alphabetical order.**

In [70]:
uniqueWords = sorted(set(preprocessed))
print(len(uniqueWords)) #2944 unique words

1139


In [71]:
vocab = {token:integer for integer, token in enumerate(uniqueWords)} #token-integer pairs made
for i, item in enumerate(vocab.items()):
    print(item) #we show the first 20 pairs as a check
    if i >= 20: break

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('-', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)


**-We also want the ability to convert IDs back to text. Lets make a class to both encode text and decode IDs.**

In [72]:
class SimpleTokenizerV1:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\'-]|--|\s)', text)
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'-])', r'\1', text)
        return text

**We test the abilities of the tokenizer with a small string.**

In [73]:
text = "I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera. (Though I rather thought it would have been Rome or Florence.)"


tokenizer = SimpleTokenizerV1(vocab)
ids = tokenizer.encode(text)
print(ids)

print(tokenizer.decode(ids))


[53, 44, 149, 1012, 57, 38, 825, 115, 258, 492, 6, 6, 1011, 115, 506, 440, 397, 6, 6, 916, 588, 1087, 713, 512, 970, 1025, 666, 1025, 537, 996, 5, 571, 997, 540, 726, 551, 502, 5, 535, 518, 375, 551, 752, 5, 664, 115, 848, 1111, 5, 157, 402, 549, 571, 115, 1076, 731, 997, 84, 7, 3, 99, 53, 825, 1012, 588, 1129, 532, 206, 85, 738, 34, 7, 4]
I HAD always thought Jack Gisburn rather a cheap genius-- though a good fellow enough-- so it was no great surprise to me to hear that, in the height of his glory, he had dropped his painting, married a rich widow, and established himself in a villa on the Riviera.( Though I rather thought it would have been Rome or Florence.)


**-We need to add special tokens which indicate unknown words, and end of text. We'll make another tokenizer SimpleTokenizerV2:**

In [74]:
#we add two special tokens endoftext and unk  into our tokens and add it to our vocab
all_tokens = sorted(list(set(preprocessed)))
all_tokens.extend(["<|endoftext|>", "<|unk|>"])
vocab = {token:integer for integer, token in enumerate(all_tokens)}
for i, item in enumerate(list(vocab.items())[-5:]):
    print(item)

('younger', 1136)
('your', 1137)
('yourself', 1138)
('<|endoftext|>', 1139)
('<|unk|>', 1140)


In [75]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = { i:s for s,i in vocab.items()}
    
    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        preprocessed = [item.strip() for item in preprocessed if item.strip()]
        preprocessed = [
            item if item in self.str_to_int 
            else "<|unk|>" for item in preprocessed
        ]

        ids = [self.str_to_int[s] for s in preprocessed]
        return ids
        
    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.:;?!"()\'])', r'\1', text)
        return text

In [76]:
#trying out the new tokenizer which utilizes unknown word and endoftext tokens
text1 = "one two three four five six"
text2 = "a b c d e f"
text = " <|endoftext|> ".join((text1, text2))
print(text)

#print results
tokenizer = SimpleTokenizerV2(vocab)
print(tokenizer.encode(text))
print(tokenizer.decode(tokenizer.encode(text)))

#Also consider other special tokens such as beginning of sequence BOS, end of sequence EOS and padding PAD tokens

one two three four five six <|endoftext|> a b c d e f
[733, 1140, 1013, 476, 450, 1140, 1139, 115, 1140, 1140, 315, 1140, 1140]
one <|unk|> three four five <|unk|> <|endoftext|> a <|unk|> <|unk|> d <|unk|> <|unk|>


**-2.5: Byte Pair Encoding**

We'll use tiktoken to avoid the complexity of byte pair encoding

In [78]:
from importlib.metadata import version
import tiktoken
print(version("tiktoken"))

0.14.0


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")


[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]


**-Example usage of the BPE tokenizer**

Note that the BPE already has a vocab of 50,257 words, and for unknown words it breaks it down into substrings (down to the character) and encodes the substrings.

In [81]:
tokenizer = tiktoken.get_encoding("gpt2")
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     "of someunknownPlace."
)

integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

print(integers)

strings = tokenizer.decode(integers)
print(strings)

[15496, 11, 466, 345, 588, 8887, 30, 220, 50256, 554, 262, 4252, 18250, 8812, 2114, 1659, 617, 34680, 27271, 13]
Hello, do you like tea? <|endoftext|> In the sunlit terracesof someunknownPlace.
